# triangle-barycentric — worked example 2: Convert Cartesian points to barycentric and test inside

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `triangle-barycentric`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

To check whether an arbitrary 2D point lies inside a triangle, you first convert from Cartesian to barycentric coordinates using the 2×2 edge-basis solve `[B-A | C-A] @ [u, v] = P - A`, then apply the inside predicate `u >= 0, v >= 0, u + v <= 1`. The two steps compose cleanly: `t.linalg.solve` handles the coordinate change, and the boolean test handles the containment check.

## Worked solution

**Step 1 – Define a non-standard triangle.** We use `A = (1, 0)`, `B = (4, 1)`, `C = (2, 3)` to move away from the canonical simplex.

**Step 2 – Build the edge matrix.** `M = t.stack([B - A, C - A], dim=1)` is a 2×2 matrix whose columns are the two edge vectors. This is the correct arrangement because we want `M @ [u, v] = P - A`.

**Step 3 – Solve for a batch of points.** We have several query points `Ps` of shape `(K, 2)`. We solve `M @ [u, v]^T = (P - A)^T` for all points simultaneously: `uvs = t.linalg.solve(M, (Ps - A).T).T`.

**Step 4 – Apply inside predicate.** `u, v = uvs[:, 0], uvs[:, 1]`. The mask is `(u >= 0) & (v >= 0) & (u + v <= 1)`. We verify with a known centroid point (which must be inside) and a point far outside.

In [ ]:
import torch as t

def worked2_cartesian_to_barycentric_inside():
    """
    Convert several Cartesian points to barycentric coords w.r.t. a triangle,
    then test which are inside.
    Returns (uvs, inside_mask).
    """
    A = t.tensor([1.0, 0.0])
    B = t.tensor([4.0, 1.0])
    C = t.tensor([2.0, 3.0])

    # Edge matrix: columns are B-A and C-A
    M = t.stack([B - A, C - A], dim=1)  # (2, 2)

    # Query points (centroid must be inside; far point must be outside)
    centroid = (A + B + C) / 3.0
    outside_pt = t.tensor([10.0, 10.0])
    edge_midpoint = (A + B) / 2.0  # on edge AB, should be on boundary (inside)
    extra_pt = A - (B - A)  # reflection of B through A, outside

    Ps = t.stack([centroid, outside_pt, edge_midpoint, extra_pt])  # (4, 2)

    # Solve: M @ [u, v]^T = (P - A)^T  for all points simultaneously
    # (Ps - A).T is (2, K); solve gives (2, K); transpose back to (K, 2)
    uvs = t.linalg.solve(M, (Ps - A).T).T   # (4, 2)

    u, v = uvs[:, 0], uvs[:, 1]
    inside = (u >= 0) & (v >= 0) & (u + v <= 1)

    return uvs, inside

uvs, inside = worked2_cartesian_to_barycentric_inside()
labels = ['centroid', 'far outside', 'edge midpoint', 'reflected outside']
for label, uv, flag in zip(labels, uvs, inside):
    print(f'{label}: u={uv[0]:.3f}, v={uv[1]:.3f}, inside={flag.item()}')